[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DBertazioli/aml-labs/blob/main/lab1/1_toy_keras.ipynb)

<img src="https://upload.wikimedia.org/wikipedia/commons/a/ae/Keras_logo.svg" alt="Keras" height="70">

# Lab 1, notebook 1: Keras basics and toy data

**Advanced Machine Learning, UNIMIB.**

Goal: learn the Keras workflow (build a model, compile, fit, evaluate) on the circles data. Then train Keras classifiers on four small synthetic datasets (two in 2-D, two in 3-D). See what the output activation, the loss, the width and depth, and the optimizer each change.

### Documentation

- [Getting started](https://keras.io/getting_started/) and the [API reference](https://keras.io/api/)
- Guides: [the Sequential model](https://keras.io/guides/sequential_model/), [the Functional API](https://keras.io/guides/functional_api/), [training and evaluation with `fit`](https://keras.io/guides/training_with_built_in_methods/)

### How to use this notebook

- Run the cells from top to bottom; a CPU runtime is enough.
- Ten cells are marked **TODO**: replace `None` or `# TODO` with your code, then run the check cell below it.
- Sections 2-9: synthetic data, Keras basics, then output layers, losses, model size, the optimizer and common pitfalls.
- Sections 10-11 are theory recaps for the demos we run together in class: optimizers ([optimizer race](https://dbertazioli.github.io/aml-labs/lab1/race3d/)) and depth and initialization ([going deep](https://dbertazioli.github.io/aml-labs/lab1/deep/)). They have no TODOs.
- Solutions are released after the lab.

## 1. Setup

`set_seed` fixes the random seeds, so a re-run gives the same numbers on the same setup.

*Further reading: [Keras docs, set_random_seed](https://keras.io/api/utils/python_utils/#setrandomseed-function)*

In [ ]:
import numpy as np
import keras
from keras import layers

def set_seed(s):
    keras.utils.set_random_seed(s)

set_seed(0)
print("keras", keras.__version__, "| backend:", keras.backend.backend())

## 2. Synthetic data

The notebook uses four synthetic datasets, made by numpy generators with a fixed seed:

- `circles(n)`: class 0 is a disk, class 1 a ring around it (Gaussian noise, sd 0.20). Sections 2 to 5 use it.
- `two_spirals(n)`: two arms winding around each other (noise sd 0.085).
- `ring_chain(n)`: 3-D, three rings linked like a chain (3 classes, noise sd 0.08). The middle ring passes through the holes of the other two.
- `spiral_arms_3d(n)`: 3-D, three spiral arms rising along the z axis (3 classes, noise sd 0.05).

![The four datasets](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/data_overview.png)

*The four datasets. Colours are the classes, the same colours as in the plots below.*

Conventions:

- `n` is the number of points **per class**.
- `split` keeps 80% of the points for training and 20% for test.
- Labels `y` are integers (`0`, `1`, `2`).

The cell below defines the generators, `split`, the plot helpers and `LivePlot`, the plot that updates during training. It also defines two small Keras helpers, `probs` and the callback `KerasLive` (both explained in section 3, under Training). You do not need to read it.

In [ ]:
import numpy as np

def _shuffle(X, y, rng):
    p = rng.permutation(len(y))
    return X[p].astype("float32"), y[p].astype("int64")

def circles(n, noise=0.20, seed=1):
    rng = np.random.default_rng(seed)
    r = np.r_[0.5 * np.sqrt(rng.random(n)), np.ones(n)]
    t = 2 * np.pi * rng.random(2 * n)
    X = np.c_[r * np.cos(t), r * np.sin(t)] + noise * rng.standard_normal((2 * n, 2))
    return _shuffle(X, np.r_[np.zeros(n), np.ones(n)], rng)

def two_spirals(n, noise=0.085, seed=4):
    rng = np.random.default_rng(seed)
    k = np.r_[np.zeros(n), np.ones(n)]
    r = 0.1 + 0.9 * rng.random(2 * n)
    t = r * 3 * np.pi + k * np.pi
    X = np.c_[r * np.cos(t), r * np.sin(t)] + noise * rng.standard_normal((2 * n, 2))
    return _shuffle(X, k, rng)

def _ring(rng, m, c, e1, e2, R, noise):
    t = 2 * np.pi * rng.random(m)
    return (np.array(c) + R * (np.cos(t)[:, None] * e1 + np.sin(t)[:, None] * e2)
            + noise * rng.standard_normal((m, 3)))

def ring_chain(n, noise=0.08, seed=12):
    rng = np.random.default_rng(seed)
    X, Y, Z = np.eye(3)
    rings = [_ring(rng, n, [0, 0, 0], X, Y, 1, noise),        # class 0
             _ring(rng, n, [1.3, 0, 0], X, Z, 1.3, noise),    # class 1 passes through both
             _ring(rng, n, [2.6, 0, 0], X, Y, 1, noise)]      # class 2
    return _shuffle(np.vstack(rings), np.repeat([0, 1, 2], n), rng)

def spiral_arms_3d(n, noise=0.05, seed=11):
    rng = np.random.default_rng(seed)
    k = np.repeat([0, 1, 2], n)
    h = rng.random(3 * n)
    r, t = 0.2 + 0.8 * h, k * 2 * np.pi / 3 + 5 * h
    X = np.c_[r * np.cos(t), r * np.sin(t), 2 * h - 1] + noise * rng.standard_normal((3 * n, 3))
    return _shuffle(X, k, rng)

def split(X, y, test=0.2, seed=0):
    p = np.random.default_rng(seed).permutation(len(y))
    m = int(len(y) * test)
    return X[p[m:]], y[p[m:]], X[p[:m]], y[p[:m]]


import matplotlib.pyplot as plt
import plotly.graph_objects as go

COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]

def to_labels(p):
    """probabilities (n, 1) or (n, K) -> integer labels"""
    return (p[:, 0] > 0.5).astype(int) if p.shape[1] == 1 else p.argmax(1)

def plot_data(X, y, title=""):
    plt.figure(figsize=(4, 4))
    plt.scatter(X[:, 0], X[:, 1], c=[COLORS[k] for k in y], s=8)
    plt.gca().set_aspect("equal"); plt.title(title); plt.show()

def plot_boundary(model, X, y, title=""):
    lo, hi = X.min(0) - 0.2, X.max(0) + 0.2
    xx, yy = np.meshgrid(np.linspace(lo[0], hi[0], 150), np.linspace(lo[1], hi[1], 150))
    grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
    z = to_labels(probs(model, grid)).reshape(xx.shape)
    plt.figure(figsize=(4, 4))
    plt.contourf(xx, yy, z, levels=[-0.5, 0.5, 1.5], colors=COLORS[:2], alpha=0.25)
    plt.scatter(X[:, 0], X[:, 1], c=[COLORS[k] for k in y], s=8)
    plt.gca().set_aspect("equal"); plt.title(title); plt.show()

def _scene(fig, title):
    fig.update_layout(title=title, height=450, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(aspectmode="data"))
    fig.show()

def plot3d(X, y, title=""):
    fig = go.Figure([go.Scatter3d(x=X[y == k, 0], y=X[y == k, 1], z=X[y == k, 2], mode="markers",
                                  marker=dict(size=2.5, color=COLORS[k]), name=f"class {k}")
                     for k in np.unique(y)])
    _scene(fig, title)

def plot3d_pred(X, y, p, title="Test points coloured by predicted class (x = error)"):
    pred = to_labels(p); bad = pred != y
    fig = go.Figure([go.Scatter3d(x=X[pred == k, 0], y=X[pred == k, 1], z=X[pred == k, 2], mode="markers",
                                  marker=dict(size=3, color=COLORS[k]), name=f"predicted {k}")
                     for k in np.unique(y)])
    fig.add_trace(go.Scatter3d(x=X[bad, 0], y=X[bad, 1], z=X[bad, 2], mode="markers",
                               marker=dict(size=5, color="black", symbol="x"), name="errors"))
    _scene(fig, title + f": {bad.sum()} errors of {len(y)}")

import time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

class LivePlot:
    """Keeps the per-epoch values and redraws the curves while training runs.
    update(logs) adds one epoch (redraw at most every `every` seconds); draw() shows the current state."""
    def __init__(self, every=0.5):
        self.h, self.every, self.t, self.title = {}, every, time.time(), ""   # h: (run, name) -> values per epoch
    def update(self, logs, run=""):                   # run: curve name when one plot shows several runs
        for k, v in logs.items():
            self.h.setdefault((run, k), []).append(float(v))
        n = len(self.h[(run, k)])
        self.title = f"{run + ': ' if run else ''}epoch {n} | " + " | ".join(f"{k} {float(v):.3f}" for k, v in logs.items())
        if time.time() - self.t > self.every:
            self.draw()
    def draw(self):
        if not self.h:
            return
        panels = [p for p in ("loss", "accuracy") if any(p in k for _, k in self.h)]
        several = len({run for run, _ in self.h}) > 1   # legend outside the axes when several runs are drawn
        fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels) + 1.7 * several, 3), squeeze=False)
        for ax, p in zip(axes[0], panels):
            for (run, k), v in self.h.items():
                if p in k:
                    ax.plot(range(1, len(v) + 1), v, label=run or k)
            ax.set_xlabel("epoch"); ax.set_title(p)
            if several:
                ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
            else:
                ax.legend()
        fig.suptitle(self.title); fig.tight_layout()
        clear_output(wait=True); display(fig); plt.close(fig)
        self.t = time.time()

def probs(model, A):
    return np.asarray(model(A, training=False))

class KerasLive(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.live = LivePlot()
    def on_epoch_end(self, epoch, logs=None):
        self.live.update(logs)
    def on_train_end(self, logs=None):
        self.live.draw()

### Data as numpy arrays

Keras takes numpy arrays directly in `fit`, `evaluate` and `predict`:

- Inputs: a `float32` array of shape `(n, d)`, one row per point. Here `d = 2`.
- Labels: integers, shape `(n,)`. Here 0 and 1.
- No preprocessing is needed here: the coordinates are roughly between -2 and 2.

For bigger data Keras also accepts a `tf.data.Dataset`. We do not need it.

The data below are the circles: 500 points per class, 800 for training and 200 for test.

In [ ]:
X, y = circles(500)
X_tr, y_tr, X_te, y_te = split(X, y)
assert X_tr.dtype == "float32" and X_tr.shape == (800, 2)
assert y_tr.shape == (800,) and set(np.unique(y_tr)) == {0, 1}
print("inputs", X_tr.shape, X_tr.dtype, "| labels", y_tr.shape, y_tr.dtype)
print("coordinate range:", X_tr.min().round(2), "to", X_tr.max().round(2), "| share of class 1:", y_tr.mean())
plot_data(X_tr, y_tr, "circles: training data")

## 3. Keras basics

Build, train and evaluate a small model on the circles data.

### Building a model

A Keras model is a stack of layers. Ours takes the 2 coordinates of a point and returns one number: the probability that the point is in class 1.

- `keras.Sequential([...])` applies the layers one after the other.
- `keras.Input(shape=(d,))` declares the number of input features. The first Dense layer needs it to create its weights.
- `layers.Dense(units, activation=...)` is a fully connected layer: it computes $\text{activation}(Wx + b)$ for `units` outputs. Its kernel $W$ has shape `(in, units)` and its bias `(units,)`.
- Without an activation, a Dense layer is linear.
- A stack of linear layers is still a single linear map, so hidden layers need a non-linear activation.

![The demo model as a network](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/keras_model.png)

*The demo model: 2 inputs, 8 hidden ReLU units, 1 sigmoid output. The activation is part of each Dense layer.*

Activations used in this notebook:

- `relu`: $\max(0, z)$
- `tanh`: $\tanh z$
- `sigmoid`: $1/(1+e^{-z})$, output in (0, 1)
- `softmax`: turns K scores into K probabilities, used for K classes

![ReLU, tanh and sigmoid](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/activations.png)

*ReLU, tanh and sigmoid for z between -4 and 4.*

### Output layer and loss for two classes

**Output layer.** One output unit gives the probability $p$ that the label is 1. The **sigmoid** $1/(1+e^{-z})$ maps any real score $z$ to (0, 1), so it is the right activation for that unit.

**Loss.** Binary cross-entropy, averaged over the samples:

$$-\big(y \log p + (1 - y) \log(1 - p)\big)$$

It is close to 0 when $p$ is near the true label. It grows without bound when the model is confidently wrong.

Accuracy is not used as a loss because it is flat almost everywhere: its gradient is 0.

*Further reading: [Deep Learning book, section 8.1.2](https://www.deeplearningbook.org/contents/optimization.html) (surrogate loss functions)*

> **TODO 1.** Choose the output activation and the loss for binary classification. The check cell below passes when it runs without an error.
>
> Write them as the strings Keras expects (for example `"relu"`); the loss name contains `crossentropy`.

In [ ]:
out_act = None  # TODO: activation name for the output unit
loss = None     # TODO: loss name

In [ ]:
assert out_act == "sigmoid", "binary output: use the sigmoid activation"
assert loss == "binary_crossentropy", "use binary cross-entropy"

The cell below builds the demo model with your output layer. `summary()` lists each layer with its output shape and its number of parameters. `None` in an output shape is the batch size.

A `Dense` layer with $n_{in}$ inputs and $n_{out}$ outputs has $n_{in} n_{out}$ weights and $n_{out}$ biases. Here the first layer has 2 × 8 + 8 = 24 parameters, the second 8 + 1 = 9, total 33.

In [ ]:
set_seed(0)
model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation=out_act),
])
model.summary()

> **TODO 2.** The model `mlp` below has two hidden layers of 16 ReLU units. Count its parameters by hand (weights plus biases of each layer) in `n_hand`, and with `mlp.count_params()` in `n_auto`. The two numbers must match.

*Further reading: [Keras docs, Model.count_params](https://keras.io/api/models/model/#count_params-method)*

In [ ]:
mlp = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation=out_act),
])
n_hand = None  # TODO: weights + biases of each layer, written as a sum
n_auto = None  # TODO: use mlp.count_params()

In [ ]:
assert n_auto == mlp.count_params(), "n_auto is not mlp.count_params()"
assert n_hand == n_auto, f"by hand {n_hand}, but Keras counts {n_auto}"

### Inside the model

`model.layers` lists the layers. Each layer keeps its variables in `layer.weights`, and `layer.get_weights()` returns them as numpy arrays.

`Dense` starts with a Glorot-uniform kernel and a zero bias. Kernel values are drawn uniformly from $[-b, b]$, where $n_{in}$ and $n_{out}$ are the layer's input and output sizes:

$$b = \sqrt{\frac{6}{n_{in} + n_{out}}}$$

A uniform distribution on $[-b, b]$ has standard deviation $b / \sqrt{3}$. The cell below compares this with the measured value for the second layer of `mlp` (16 × 16 weights).

*Further reading: [Glorot & Bengio, Understanding the difficulty of training deep feedforward neural networks (2010)](https://proceedings.mlr.press/v9/glorot10a.html), [Keras docs, Dense](https://keras.io/api/layers/core_layers/dense/) (default `glorot_uniform` kernel, zero bias)*

In [ ]:
for layer in mlp.layers:
    print(f"{type(layer).__name__:6s}", [tuple(v.shape) for v in layer.weights])
kernel, bias = mlp.layers[1].get_weights()
print("second layer: bias all zeros:", bool((bias == 0).all()))
print("kernel std:", kernel.std().round(3), "| Glorot uniform std = bound/sqrt(3) =", round((6 / (16 + 16)) ** 0.5 / 3 ** 0.5, 3))

### Training

![What model.fit does](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/keras_fit.png)

*What `model.fit` does with the circles data: 640 rows for the gradients, 160 for validation, 20 optimizer steps per epoch.*

Training takes three calls:

1. `compile` chooses the update rule, the quantity to minimize and what to report.
2. `fit` trains the model.
3. `evaluate` and `predict` score the model and return its outputs.

```python
model.compile(optimizer=..., loss=..., metrics=["accuracy"])
history = model.fit(X, y, epochs=E, batch_size=B, validation_split=0.2)
model.evaluate(X, y)   # [loss, accuracy] on the given data
model.predict(X)       # output of the last layer for every row of X
```

**compile**

- `optimizer`: here `Adam` with learning rate 0.01. Section 10 compares optimizers.
- `loss`: the string from TODO 1.
- `metrics=["accuracy"]`: for a sigmoid output, a point counts as class 1 when the probability is above 0.5.

*Further reading: [Keras docs, Model.compile](https://keras.io/api/models/model_training_apis/#compile-method), [Keras docs, BinaryAccuracy](https://keras.io/api/metrics/accuracy_metrics/#binaryaccuracy-class)*

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])

**fit**

- One **epoch** is one pass over the training data.
- The data are cut into minibatches of `batch_size` samples. The optimizer takes one step per minibatch.
- `validation_split=0.2` holds out the last 20% of the data. It is not used for the gradient, but loss and accuracy on it are reported after every epoch.
- The training samples are shuffled again at every epoch.
- `verbose=0` turns off the text progress bar. `callbacks=[KerasLive()]` draws the loss and accuracy curves, and the plot updates while training runs.

`KerasLive` is a callback: `fit` calls its methods at the start and end of training and at the end of every epoch. It passes the loss and accuracy of each epoch to `LivePlot`, which redraws the curves.

*Further reading: [Keras docs, Model.fit](https://keras.io/api/models/model_training_apis/#fit-method) (`validation_split` takes the last samples, before shuffling), [Keras guide, writing your own callbacks](https://keras.io/guides/writing_your_own_callbacks/)*

In [ ]:
h = model.fit(X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])

**History**

`fit` returns a `History` object. Its dict `history.history` has one list per quantity, with one value per epoch: `loss`, `accuracy`, `val_loss`, `val_accuracy`.

The training values are averages over the epoch's minibatches, while the weights are still changing. The validation values are measured once, at the end of the epoch. So in the first epochs the training curve can look worse than the validation curve.

*Further reading: [Keras FAQ, why is my training loss much higher than my testing loss?](https://keras.io/getting_started/faq/#why-is-my-training-loss-much-higher-than-my-testing-loss)*

In [ ]:
print(list(h.history))
print("last epoch:", {k: round(v[-1], 3) for k, v in h.history.items()})

**evaluate and predict**

- `evaluate(X, y)` returns the loss, then the metrics, in the order given to `compile`.
- `predict(X)` returns the output of the last layer for every row: here the probability of class 1, shape `(n, 1)`.

Both run in inference mode. The test set was not used for training or validation. `probs(model, A)`, defined in the helper cell, does what `predict` does without the progress bar; the plot helpers use it.

In [ ]:
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
p = model.predict(X_te, verbose=0)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f} | predict output shape {p.shape}")

`evaluate` computes the accuracy for us. Here you compute it yourself from the predicted probabilities `p`: shape `(n, 1)`, the probability of class 1 for each test point.

> **TODO 3.** Turn `p` into 0/1 predictions with threshold 0.5, then compute the fraction that equals `y_te`. The result must match the `test_acc` from `evaluate`.

In [ ]:
pred = None  # TODO: 0/1 predictions, shape (n,)
acc = None   # TODO: fraction of test points with pred == y_te

In [ ]:
assert acc is not None and abs(acc - test_acc) < 1e-4, f"by hand {acc} but evaluate says {test_acc}"

**What the model learned.** The background colour is the class the model predicts at each point of a grid. The dots are the test points.

In [ ]:
plot_boundary(model, X_te, y_te, "one hidden layer of 8 units, test points")

### The Functional API (optional)

`Sequential` stacks layers in one chain. The Functional API calls layers on tensors, like functions, so a model can have several inputs, several outputs, and paths that split and merge.

Example, **wide and deep**: the input goes through two Dense layers (the deep path) and also straight to the end (the wide path). `Concatenate` joins the two, and one Dense unit gives the output.

![A wide and deep model](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/functional_wide_deep.png)

*The wide and deep model: the input feeds two paths that meet at `Concatenate`. Output shapes start with `None`, the batch size.*

`Sequential` cannot express this model. It stacks layers in one chain, and here the input feeds two paths that merge.

`keras.utils.plot_model(model, show_shapes=True)` draws such diagrams automatically. It needs pydot and Graphviz, so we do not call it here.

*Further reading: [Keras docs, plot_model](https://keras.io/api/utils/model_plotting_utils/), [Keras guide, the Functional API](https://keras.io/guides/functional_api/), [Cheng et al., Wide & Deep Learning for Recommender Systems (2016)](https://arxiv.org/abs/1606.07792)*

In [ ]:
set_seed(0)
inputs = keras.Input(shape=(2,))
deep = layers.Dense(16, activation="relu")(inputs)
deep = layers.Dense(16, activation="relu")(deep)
merged = layers.Concatenate()([inputs, deep])    # the wide path is the raw inputs
outputs = layers.Dense(1, activation=out_act)(merged)
wide_deep = keras.Model(inputs, outputs)
wide_deep.summary()

The "Connected to" column of `summary()` shows which layers each layer reads from. `concatenate` reads from the input and from the second Dense layer.

The next cells train the wide and deep model and a plain MLP with the same two hidden layers, on the circles.

In [ ]:
wide_deep.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])
h_wd = wide_deep.fit(X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])

In [ ]:
set_seed(0)
plain = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation=out_act),
])
plain.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])
h_plain = plain.fit(X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])

In [ ]:
for name, m in [("plain MLP", plain), ("wide and deep", wide_deep)]:
    print(f"{name:13s} | parameters {m.count_params()} | test accuracy {m.evaluate(X_te, y_te, verbose=0)[1]:.3f}")

The wide path adds only 2 parameters: one weight from each raw input to the output unit.

Both models fit the circles about equally well: on 200 test points, a difference of a few points of accuracy is noise.

## 4. Sanity checks before real training

Two quick checks catch most bugs before a long training run. Both use the circles data from section 2.

### Loss at initialization

At initialization the weights are small and random, so the output is a probability near 0.5 for every point. The cross-entropy is then close to $\ln 2 \approx 0.69$.

A value far from that points to a bug: bad initialization, unscaled inputs or the wrong loss.

*Further reading: [Karpathy, A Recipe for Training Neural Networks (2019)](https://karpathy.github.io/2019/04/25/recipe/), step "verify loss @ init"*

> **TODO 4.** Compute `init_loss`: the mean binary cross-entropy of the untrained model's probabilities `p0` against `y_tr`.
>
> Use `keras.losses.BinaryCrossentropy()`: create the loss object, then call it as `loss_fn(y_true, y_pred)`.

In [ ]:
set_seed(0)
model0 = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation=out_act),
])
p0 = model0.predict(X_tr, verbose=0)   # probabilities of the untrained model, shape (n, 1)
init_loss = None  # TODO: mean binary cross-entropy of p0 against y_tr
print("init loss:", init_loss, "| ln 2 =", round(float(np.log(2)), 3))

In [ ]:
assert init_loss is not None, "compute init_loss"
assert abs(init_loss - np.log(2)) < 0.15, f"init loss {init_loss:.3f} is far from ln 2 = {np.log(2):.3f}"

### Overfit one small batch

A correct model and optimizer drive the loss on 32 samples close to 0. If they cannot, there is a bug.

*Further reading: [Karpathy, A Recipe for Training Neural Networks (2019)](https://karpathy.github.io/2019/04/25/recipe/), step "overfit one batch"; [Deep Learning book, section 11.5](https://www.deeplearningbook.org/contents/guidelines.html) ("fit a tiny dataset")*

> **TODO 5.** Overfit one batch: fit `m1` on `xb, yb` for 300 epochs with `batch_size=32`. That is one batch per epoch, so 300 optimizer steps on the same 32 samples.
>
> Add `verbose=0, callbacks=[KerasLive()]`. Store the `History` in `h` and the loss after the last epoch in `final_loss`.

In [ ]:
set_seed(0)
xb, yb = X_tr[:32], y_tr[:32]
m1 = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation=out_act),
])
m1.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss)
h = None           # TODO: m1.fit(...) on xb, yb
final_loss = None  # TODO: training loss after the last epoch

In [ ]:
assert h is not None and len(h.history["loss"]) == 300, "train for 300 epochs"
assert int(m1.optimizer.iterations) == 300, "one batch of 32 per epoch: use batch_size=32"
assert final_loss < 0.01, f"final loss {final_loss:.4f} is not close to 0"

## 5. Circles: a straight line is not enough

Same circles data as in section 2.

**Logistic regression** is a model with no hidden layer: one output unit with a sigmoid on top of a linear function of the inputs.

Its decision boundary is a straight line. No straight line separates a disk from a ring around it.

In [ ]:
set_seed(0)
model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(1, activation=out_act),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])
h = model.fit(X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot_boundary(model, X_te, y_te, "logistic regression, test points")

Accuracy stays low: a straight line does little better than the 50% chance level of two balanced classes.

Now the model of section 3 again: **one hidden layer** with 8 ReLU units. Each unit is a linear function followed by $\max(0, z)$. The output unit combines them, so the boundary can bend.

The seed is the same, so the numbers are the same as in section 3.

In [ ]:
set_seed(0)
model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation=out_act),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])
h = model.fit(X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot_boundary(model, X_te, y_te, "one hidden layer, test points")

## 6. Two spirals: capacity

Same output layer and loss, harder data: the two arms wind around each other.

The first model below is small: one hidden layer of 4 units, 30 epochs. It underfits for two reasons:

- It cannot represent the boundary.
- It has few optimizer steps: 30 epochs are 600 steps (20 minibatches of 32 per epoch).

In [ ]:
X, y = two_spirals(500)
X_tr, y_tr, X_te, y_te = split(X, y)
plot_data(X_tr, y_tr, "two spirals: training data")

In [ ]:
set_seed(0)
model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(4, activation="relu"),
    layers.Dense(1, activation=out_act),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])
h = model.fit(X_tr, y_tr, epochs=30, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot_boundary(model, X_te, y_te, "small model, test points")

### Capacity

The **capacity** of a model is set by its units per layer and its number of layers.

The noise also limits the result: the arms overlap a little, so the best possible test accuracy is about 0.95, not 1.0.

> **TODO 6.** Change `hidden` (list of hidden layer widths, for example `[16, 16]`) and `epochs` so that the **test accuracy is at least 0.88**.
>
> Training should take well under a minute. If the target is missed, try a wider or deeper network first, then more epochs.

In [ ]:
set_seed(0)
hidden = [4]   # TODO: widths of the hidden layers, e.g. [16, 16]
epochs = 30    # TODO: number of epochs

model = keras.Sequential([keras.Input(shape=(2,))]
                         + [layers.Dense(u, activation="relu") for u in hidden]
                         + [layers.Dense(1, activation=out_act)])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])
h = model.fit(X_tr, y_tr, epochs=epochs, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot_boundary(model, X_te, y_te, "test points")

In [ ]:
TARGET = 0.88
assert test_acc >= TARGET, f"test accuracy {test_acc:.3f} is below {TARGET}: increase hidden and/or epochs"

## 7. Ring chain: K classes

Three classes, each one a ring in 3-D, linked like a chain. No plane separates one ring from the others.

The plot is interactive: rotate it.

In [ ]:
X, y = ring_chain(400)
X_tr, y_tr, X_te, y_te = split(X, y)
plot3d(X_tr, y_tr, "ring chain: training data")

### Output layer for K classes

The last layer has K units, one per class. **Softmax** turns the K raw scores $z_1, \dots, z_K$ into probabilities:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

The probabilities are positive and add up to 1. The predicted class is the one with the largest probability.

### Integer labels vs one-hot

Our `y` holds integers 0, 1, 2. One-hot encoding writes the same labels as `[1,0,0]`, `[0,1,0]`, `[0,0,1]`.

Categorical cross-entropy is $-\log p_c$, where $c$ is the true class.

- With one-hot labels this is `categorical_crossentropy` in Keras.
- With integer labels it is `sparse_categorical_crossentropy`.
- Both compute the same number. Integer labels need no conversion, so we use them.

**In Keras:**

- The output layer is `layers.Dense(K, activation="softmax")`.
- The loss is the string `"sparse_categorical_crossentropy"`.

*Further reading: [Keras docs, SparseCategoricalCrossentropy](https://keras.io/api/losses/probabilistic_losses/#sparsecategoricalcrossentropy-class)*

> **TODO 7.** Define the output layer for 3 classes (the hidden layer below has 16 units) and the loss. Name the loss `loss3`, so that the binary `loss` from TODO 1 stays valid.

In [ ]:
out_layer = None  # TODO: the output layer for 3 classes
loss3 = None      # TODO: loss name for integer labels

In [ ]:
assert isinstance(out_layer, layers.Dense) and out_layer.units == 3, "3 classes need 3 output units"
assert out_layer.activation.__name__ == "softmax", "use softmax"
assert loss3 == "sparse_categorical_crossentropy", "integer labels: sparse categorical cross-entropy"

In [ ]:
set_seed(0)
model = keras.Sequential([
    keras.Input(shape=(3,)),
    layers.Dense(16, activation="relu"),
    layers.Dense.from_config(out_layer.get_config()),   # a fresh copy of your layer: re-running this cell starts from new weights
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss3, metrics=["accuracy"])
h = model.fit(X_tr, y_tr, epochs=40, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot3d_pred(X_te, y_te, probs(model, X_te))

## 8. Spiral arms: change only the optimizer

Three arms rise along the z axis, each winding around it.

In [ ]:
X, y = spiral_arms_3d(400)
X_tr, y_tr, X_te, y_te = split(X, y)
plot3d(X_tr, y_tr, "spiral arms: training data")

The model below is fixed: one hidden layer of 16 units, 40 epochs, trained with SGD at learning rate 0.01. It reaches a test accuracy of only about 0.6.

Section 10 lists the optimizer options.

> **TODO 8.** Change **only the optimizer settings** (the marked line) so that the **test accuracy is at least 0.95**.

In [ ]:
set_seed(0)
model = keras.Sequential([
    keras.Input(shape=(3,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(3, activation="softmax"),
])
optimizer = keras.optimizers.SGD(learning_rate=0.01)  # TODO: change only the optimizer settings
model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h = model.fit(X_tr, y_tr, epochs=40, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")

In [ ]:
TARGET = 0.95
assert test_acc >= TARGET, f"test accuracy {test_acc:.3f} is below {TARGET}: change the optimizer"

In [ ]:
plot3d_pred(X_te, y_te, probs(model, X_te))

### Learning rate: SGD and Adam

The learning rate matters more than almost any other setting.

The next cell trains the same model on the spiral arms with 7 optimizer settings, 20 epochs each. One plot shows the training loss of every setting. SGD here is plain SGD, without momentum.

The same learning rate $\eta$ means a different step size for SGD and for Adam (see "Adam's step size" in the collapsed "Advanced" subsection of section 10), so their useful ranges differ.

*Further reading: [Deep Learning book, section 11.4.1](https://www.deeplearningbook.org/contents/guidelines.html), [Bengio, Practical recommendations for gradient-based training of deep architectures (2012)](https://arxiv.org/abs/1206.5533), [Kingma & Ba, Adam: a method for stochastic optimization (2014)](https://arxiv.org/abs/1412.6980), [Deep Learning book, section 8.5](https://www.deeplearningbook.org/contents/optimization.html), [Keras docs, SGD](https://keras.io/api/optimizers/sgd/) and [Adam](https://keras.io/api/optimizers/adam/)*

`KerasLiveRun` is a variant of `KerasLive` for several runs: all of them share one `LivePlot`, and each run draws its own curve, named by `run`.

In [ ]:
class KerasLiveRun(keras.callbacks.Callback):
    def __init__(self, live, run):
        super().__init__()
        self.live, self.run = live, run
    def on_epoch_end(self, epoch, logs=None):
        self.live.update(logs, run=self.run)
    def on_train_end(self, logs=None):
        self.live.draw()

X, y = spiral_arms_3d(400)
X_tr, y_tr, X_te, y_te = split(X, y)
settings = {
    "SGD lr=0.001":   lambda: keras.optimizers.SGD(learning_rate=0.001),
    "SGD lr=0.01":    lambda: keras.optimizers.SGD(learning_rate=0.01),
    "SGD lr=0.1":     lambda: keras.optimizers.SGD(learning_rate=0.1),
    "Adam lr=0.0001": lambda: keras.optimizers.Adam(learning_rate=0.0001),
    "Adam lr=0.001":  lambda: keras.optimizers.Adam(learning_rate=0.001),
    "Adam lr=0.01":   lambda: keras.optimizers.Adam(learning_rate=0.01),
    "Adam lr=0.1":    lambda: keras.optimizers.Adam(learning_rate=0.1),
}
live = LivePlot()
rows = {}
for name, make_optimizer in settings.items():
    set_seed(0)
    m = keras.Sequential([
        keras.Input(shape=(3,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(3, activation="softmax"),
    ])
    m.compile(optimizer=make_optimizer(), loss="sparse_categorical_crossentropy")
    hist = m.fit(X_tr, y_tr, epochs=20, batch_size=32, verbose=0, callbacks=[KerasLiveRun(live, name)])
    rows[name] = {"final_loss": hist.history["loss"][-1], "test_acc": (to_labels(probs(m, X_te)) == y_te).mean()}

import pandas as pd
df = pd.DataFrame(rows).T.round(3)
df

> **TODO 9.** Using `df`, store in `best` the name (the index label) of the setting with the highest test accuracy.
>
> Then think about this: `SGD lr=0.001` and `Adam lr=0.001` use the same learning rate but end far apart. Why do SGD and Adam need different learning-rate ranges?

In [ ]:
best = None  # TODO

In [ ]:
assert best in df.index and df.loc[best, "test_acc"] == df["test_acc"].max(), "best must be the name (index label) of a setting with the highest test_acc"

## 9. Pitfalls

These mistakes usually raise no error: the model trains, but badly.

### Logits and probabilities

Our binary models end with a sigmoid, so they output probabilities. The other common design is a last layer with **no activation**. It outputs a raw score called a **logit**.

A logit is the score before the sigmoid, any real number. $\sigma(0) = 0.5$, $\sigma(2) \approx 0.88$ and $\sigma(-2) \approx 0.12$. The name comes from $\mathrm{logit}(p) = \log\frac{p}{1-p}$, the inverse of the sigmoid.

With a logit output:

- The loss must be told: `BinaryCrossentropy(from_logits=True)`, or `SparseCategoricalCrossentropy(from_logits=True)` for K classes.
- The loss then applies the sigmoid (or softmax) itself, which is more stable numerically than a separate layer.
- The accuracy needs threshold 0: `BinaryAccuracy(threshold=0.0)`. Its default, 0.5, is for probabilities.

Mixing the two up still trains, but badly. The loss treats logits as probabilities (or the reverse), and the accuracy compares a logit with 0.5.

*Further reading: [TensorFlow docs, sigmoid_cross_entropy_with_logits](https://www.tensorflow.org/api_docs/python/tf/nn/sigmoid_cross_entropy_with_logits) (the stable formula), [PyTorch docs, BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) (the same idea), [Deep Learning book, section 4.1](https://www.deeplearningbook.org/contents/numerical.html) (overflow and underflow), [Keras docs, BinaryCrossentropy](https://keras.io/api/losses/probabilistic_losses/#binarycrossentropy-class)*

The cell below computes the loss and the accuracy for four points, with the right and the wrong settings.

In [ ]:
target = np.array([1, 0, 1, 0])
logits = np.array([[2.0], [-1.0], [0.3], [-4.0]], dtype="float32")   # all four classified correctly by z > 0
prob = 1 / (1 + np.exp(-logits))
bce, bce_logits = keras.losses.BinaryCrossentropy(), keras.losses.BinaryCrossentropy(from_logits=True)
print("probabilities, from_logits=False:", float(bce(target, prob)))
print("logits, from_logits=True:        ", float(bce_logits(target, logits)), "(same loss, computed from the logits)")
print("logits, from_logits=False:       ", float(bce(target, logits)), "(wrong)")
print("probabilities, from_logits=True: ", float(bce_logits(target, prob)), "(wrong)")
for thr in (0.0, 0.5):
    print(f"BinaryAccuracy(threshold={thr}) on logits:", float(keras.metrics.BinaryAccuracy(threshold=thr)(target, logits)))

A model with a logit output, trained the right way, on the circles. It reaches about the same accuracy as the sigmoid model of section 3.

In [ ]:
X, y = circles(500)
X_tr, y_tr, X_te, y_te = split(X, y)

set_seed(0)
logit_model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1),                      # no activation: the output is a logit
])
logit_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
                    loss=keras.losses.BinaryCrossentropy(from_logits=True),
                    metrics=[keras.metrics.BinaryAccuracy(threshold=0.0)])
logit_model.fit(X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2, verbose=0, callbacks=[KerasLive()])
test_loss, test_acc = logit_model.evaluate(X_te, y_te, verbose=0)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")

### Training mode and inference mode

Dropout and BatchNorm behave differently in training and at inference.

- `fit` runs them in training mode. `predict` and `evaluate` run them in inference mode.
- If you call a model or a layer yourself, pass `training=True` or `training=False`.

Dropout sets a fraction of its inputs to 0 in training mode and does nothing at inference. The main models of this notebook do not use it. Regularization is the subject of Lab 2.

*Further reading: [Keras docs, Dropout](https://keras.io/api/layers/regularization_layers/dropout/), [Srivastava et al., Dropout (2014)](https://jmlr.org/papers/v15/srivastava14a.html)*

In [ ]:
d = layers.Dropout(0.5)
v = np.ones((1, 8), dtype="float32")
print("training=True: ", d(v, training=True).numpy())
print("training=False:", d(v, training=False).numpy())

> **TODO 10.** Collect 1000 outputs of `d` on `v` in **training** mode: `outs`, a numpy array of shape `(1000, 8)`. Then compute `train_mean`, the mean of all its values. Predict it before you run: each output above is a mix of 0 and 2.
>
> The check passes when `train_mean` is close to the inference output, 1.0.

In [ ]:
outs = None        # TODO: 1000 training-mode outputs of d on v, shape (1000, 8)
train_mean = None  # TODO: the mean of all the values in outs

In [ ]:
assert outs is not None and train_mean is not None, "compute outs and train_mean"
assert outs.shape == (1000, 8), f"outs has shape {outs.shape}, expected (1000, 8)"
assert (outs == 0).any(), "no value was set to 0: pass training=True"
assert abs(train_mean - 1.0) < 0.05, f"the mean is {train_mean:.3f}, expected about 1.0"

In training mode, Dropout multiplies the kept values by $1/(1-p) = 2$. The average output is then the same as the inference output, so the next layer sees the same scale in training and at inference.

*[Srivastava et al., Dropout (2014)](https://jmlr.org/papers/v15/srivastava14a.html)*

### Other pitfalls

- **Reusing a compiled model.** Calling `fit` again continues from the current weights. For a fresh experiment, build a new model and call `set_seed` first.
- **Shapes and dtypes.** Inputs are `float32` with shape `(n, d)`. Binary labels have shape `(n,)`. The model output has shape `(n, 1)`.
- **Integer or one-hot labels.** The loss must match the labels you have (section 7).
- **Not normalizing the inputs.** Features on very different scales make training harder. The toy datasets need no scaling; the data of notebook 2 has columns in large units.

## 10. Optimizer recap

### Notation

- $\theta$: all the weights.
- $g$: gradient of the minibatch loss at $\theta$.
- $\eta$: learning rate.
- $t$: step number.

Every operation acts per coordinate: $g^2$ squares each entry of $g$.

### Update rules

**SGD**

$$\theta \leftarrow \theta - \eta\, g$$

The step is proportional to the gradient.

**Momentum** ($\beta \approx 0.9$)

$$v \leftarrow \beta v + g$$

$$\theta \leftarrow \theta - \eta\, v$$

$v$ is a decaying sum of past gradients: directions that keep their sign add up, directions that flip sign cancel.

*Further reading: [Sutskever et al. (2013)](https://proceedings.mlr.press/v28/sutskever13.html)*

**Nesterov**

$$v \leftarrow \beta v + g$$

$$\theta \leftarrow \theta - \eta\, (g + \beta v)$$

Momentum with the gradient taken at the look-ahead point; it reacts earlier when the step is about to overshoot.

*Further reading: [Sutskever et al. (2013)](https://proceedings.mlr.press/v28/sutskever13.html)*

**AdaGrad**

$$G \leftarrow G + g^2$$

$$\theta \leftarrow \theta - \eta\, \frac{g}{\sqrt{G} + \epsilon}$$

$G$ only grows, so every coordinate's step keeps shrinking (faster where past gradients were large), and in long runs the steps vanish.

*Further reading: [Duchi et al. (2011)](https://jmlr.org/papers/v12/duchi11a.html)*

**RMSProp** ($\rho$)

$$s \leftarrow \rho s + (1-\rho)\, g^2$$

$$\theta \leftarrow \theta - \eta\, \frac{g}{\sqrt{s} + \epsilon}$$

Like AdaGrad with a moving average, so old gradients are forgotten and steps do not vanish.

*Further reading: [Hinton, lecture 6e slides (2012)](https://www.cs.toronto.edu/~hinton/coursera/lecture6/lec6.pdf)*

**Adam** ($\beta_1 = 0.9$, $\beta_2 = 0.999$)

$$m \leftarrow \beta_1 m + (1-\beta_1)\, g$$

$$s \leftarrow \beta_2 s + (1-\beta_2)\, g^2$$

$$\hat m = \frac{m}{1-\beta_1^t}, \qquad \hat s = \frac{s}{1-\beta_2^t}$$

$$\theta \leftarrow \theta - \eta\, \frac{\hat m}{\sqrt{\hat s} + \epsilon}$$

Momentum on the gradient plus RMSProp scaling; dividing by $1-\beta^t$ corrects the bias from starting $m$ and $s$ at 0.

*Further reading: [Kingma & Ba (2014)](https://arxiv.org/abs/1412.6980)*

### Framework defaults

| Setting | Keras 3 | PyTorch |
|---|---|---|
| SGD learning rate | 0.01 | 0.001 (recent versions) |
| Adam learning rate | 0.001 | 0.001 |
| Adam / RMSProp epsilon | 1e-7 | 1e-8 |
| RMSProp decay of the squared-gradient average | `rho = 0.9` | `alpha = 0.99` |
| RMSProp learning rate | 0.001 | 0.01 |
| AdaGrad learning rate | 0.001 | 0.01 |
| AdaGrad initial accumulator `G` | 0.1 | 0 |
| AdaGrad epsilon | 1e-7 | 1e-10 |
| Momentum form | `v = m*v - lr*g`, `theta += v` | `v = m*v + g`, `theta -= lr*v` |

The two momentum forms give the same trajectory for a constant learning rate: the Keras `v` is minus `lr` times the PyTorch `v`. They differ if you change the learning rate during training.

Always set the learning rate explicitly instead of relying on defaults.

Where epsilon goes:

- Keras RMSProp and AdaGrad put it inside the square root: $g / \sqrt{s + \epsilon}$ and $g / \sqrt{G + \epsilon}$. The formulas above show it outside, as PyTorch does.
- Keras Adam puts it outside.

*Further reading: Keras [SGD](https://keras.io/api/optimizers/sgd/), [Adagrad](https://keras.io/api/optimizers/adagrad/), [RMSprop](https://keras.io/api/optimizers/rmsprop/), [Adam](https://keras.io/api/optimizers/adam/). PyTorch [SGD](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html), [Adagrad](https://pytorch.org/docs/stable/generated/torch.optim.Adagrad.html), [RMSprop](https://pytorch.org/docs/stable/generated/torch.optim.RMSprop.html), [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)*

### Optimizer constructors

Interactive demo: [3-D optimizer race](https://dbertazioli.github.io/aml-labs/lab1/race3d/).

The constructors for this framework:

In [ ]:
optimizers = {
    "SGD":      keras.optimizers.SGD(learning_rate=0.01),
    "Momentum": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "Nesterov": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    "AdaGrad":  keras.optimizers.Adagrad(learning_rate=0.01),
    "RMSProp":  keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9),
    "Adam":     keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999),
}
# pass one to model.compile(optimizer=optimizers["Adam"], ...)
KEYS = ("learning_rate", "momentum", "nesterov", "rho", "beta_1", "beta_2", "epsilon", "initial_accumulator_value")
for name, opt in optimizers.items():
    print(f"{name:9s}", {k: float(f"{v:.6g}") if isinstance(v, (float, np.floating)) else v for k, v in opt.get_config().items() if k in KEYS})

### Learning-rate schedules

So far the learning rate stayed the same for the whole run. A schedule changes it during training, usually at every optimizer step:

- **At the start** the weights are random, the gradients are large and Adam's running averages are still empty. A short **warmup** raises the learning rate linearly from about 0 to its peak over the first steps, so the first updates do not throw the weights far off.
- **At the end**, near a minimum, large steps jump back and forth across it. A smaller learning rate lets the model settle, and the loss drops again.

| Schedule | Learning rate at step $t$ | Typical use |
|---|---|---|
| Constant | $\eta$ | baseline |
| Step decay | multiplied by $\gamma$ (e.g. 0.1) at fixed steps or epochs | SGD on image models |
| Exponential | $\eta\, \gamma^{t}$ | smooth version of step decay |
| Cosine | $\eta \cdot \tfrac12 \left(1 + \cos(\pi t / T)\right)$: from $\eta$ to 0 at step $T$ | common default |
| Warmup + cosine | linear rise over the first steps, then cosine | short budgets, large batches, transformers |
| Reduce on plateau | divided by a factor when the validation loss stops improving | when you do not know $T$ in advance |

With a fixed number of steps $T$, as in the competition of notebook 2, warmup + cosine is a good first choice: the peak can be higher than a constant learning rate would allow, and the last steps are small.

*Further reading: [Loshchilov & Hutter, SGDR (2017)](https://arxiv.org/abs/1608.03983) (cosine), [Goyal et al. (2017)](https://arxiv.org/abs/1706.02677) (warmup), [Smith & Topin (2018)](https://arxiv.org/abs/1708.07120) (one-cycle). Keras [learning-rate schedules](https://keras.io/api/optimizers/learning_rate_schedules/) and [ReduceLROnPlateau](https://keras.io/api/callbacks/reduce_lr_on_plateau/). PyTorch [lr_scheduler](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate)*

The schedules in this framework, plotted over 1000 steps with a peak learning rate of 0.01:

In [ ]:
T, warm = 1000, 50
S = keras.optimizers.schedules
schedules = {
    "constant":        0.01,
    "step decay":      S.PiecewiseConstantDecay(boundaries=[300, 600], values=[0.01, 0.001, 0.0001]),
    "exponential":     S.ExponentialDecay(0.01, decay_steps=T, decay_rate=0.01),
    "cosine":          S.CosineDecay(0.01, decay_steps=T),
    "warmup + cosine": S.CosineDecay(0.0, decay_steps=T - warm, warmup_target=0.01, warmup_steps=warm),
}
steps = np.arange(T)
plt.figure(figsize=(6, 3.5))
for name, s in schedules.items():
    plt.plot(steps, [float(s(t)) if callable(s) else s for t in steps], label=name)
plt.xlabel("optimizer step"); plt.ylabel("learning rate"); plt.legend(); plt.show()

**How to use one in Keras.** Pass the schedule to the optimizer in place of a number. `fit` then updates the learning rate at every step:

```python
T = epochs * int(np.ceil(len(X_tr) / batch_size))   # total optimizer steps
lr = keras.optimizers.schedules.CosineDecay(0.0, decay_steps=T - 50, warmup_target=0.01, warmup_steps=50)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=..., metrics=["accuracy"])
model.fit(X_tr, y_tr, epochs=epochs, batch_size=batch_size)
```

Reduce on plateau is a callback instead, checked at the end of every epoch (keep a number as the learning rate):

```python
model.fit(..., validation_split=0.2,
          callbacks=[keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)])
```

### Advanced: curvature, zigzag and step size

Background for the optimizer demo. You do not need it for the TODOs.

#### Curvature and the condition number

Near a minimum the loss is approximately

$$\tfrac12\, (\theta-\theta^\ast)^T H\, (\theta-\theta^\ast)$$

The eigenvalues of $H$, from $\lambda_{\min}$ to $\lambda_{\max}$, are the curvatures along its principal directions. The **condition number** is $\kappa = \lambda_{\max} / \lambda_{\min}$.

Gradient descent multiplies the error along a direction with curvature $\lambda$ by $(1 - \eta\lambda)$ at every step. Consequences:

**Stability.** All factors must satisfy $|1 - \eta\lambda| < 1$, so $\eta < 2/\lambda_{\max}$. Above this limit the iterates diverge.

**Speed.** The slowest direction shrinks by $1 - \eta\lambda_{\min}$ per step. At $\eta \approx 1/\lambda_{\max}$ that is $1 - 1/\kappa$, so about $\kappa$ steps reduce the error by a factor $e$.

With $\kappa = 100$ you need about 100 steps for what a well-conditioned problem does in 1.

**Zigzag.** If $\eta > 1/\lambda_{\max}$, the factor $1 - \eta\lambda_{\max}$ is negative. Each step jumps over the steep direction (the sign flips) while making slow progress along the flat one.

You cannot fix this by lowering $\eta$ without making the flat direction slower still.

*Further reading: [Deep Learning book, sections 4.2 and 4.3.1](https://www.deeplearningbook.org/contents/numerical.html) (poor conditioning, the Hessian and the step size), [section 8.2.1](https://www.deeplearningbook.org/contents/optimization.html) (ill-conditioning in neural network training)*

![Learning rate limit](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/opt_lr_limit.png)

*Quadratic with largest curvature 25: learning rate 0.07 converges, 0.08 (= 2/25) neither converges nor diverges, 0.09 diverges.*

![Optimizers on an elongated valley](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/opt_zigzag.png)

*Elongated valley: gradient descent zigzags across the steep direction, momentum overshoots before settling, Adam goes more directly to the minimum.*

#### Momentum and per-coordinate scaling

**Momentum.**

- With a constant gradient, $v$ converges to $g/(1-\beta)$. The **effective learning rate is $\eta/(1-\beta)$**, 10 times larger for $\beta = 0.9$.
- Steady directions are sped up by about $1/(1-\beta)$; oscillating ones partly cancel.
- In the form above, the stability limit for momentum is $\eta\lambda_{\max} < 2(1+\beta)$.

*Further reading: [Deep Learning book, section 8.3](https://www.deeplearningbook.org/contents/optimization.html) (momentum)*

**AdaGrad, RMSProp and Adam.**

- They divide each coordinate's step by an estimate of that coordinate's gradient size.
- This **helps only when the badly scaled directions are the coordinate axes**: $H$ nearly diagonal, one weight with large gradients and another with small ones.
- If the elongated valley is rotated against the axes (large off-diagonal entries in $H$), rescaling the axes does not reduce $\kappa$. The race demo shows both cases.

*Further reading: [Deep Learning book, section 8.5](https://www.deeplearningbook.org/contents/optimization.html) (adaptive learning rates: AdaGrad, RMSProp, Adam)*

**Adam's step size.**

- If the gradient is steady, $\hat m \approx g$ and $\sqrt{\hat s} \approx |g|$, so each weight moves about $\eta$ per step whatever the scale of the gradient. With noisy gradients the step is smaller.
- So $\eta = 10^{-3}$ means weights change by roughly $10^{-3}$ per step. For SGD the change scales with the gradient.

![Valley along the axes and rotated](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/opt_rotated.png)

*Left: the valley follows the coordinate axes. Right: the same valley rotated by 45 degrees, where per-coordinate rescaling does not reduce the condition number.*

## 11. Depth recap

### Backprop multiplies one factor per layer

Backprop sends the gradient of the loss back through the layers. Each layer multiplies it by one factor (its Jacobian, $J_l$).

With $L$ layers, the gradient at the input is $J_1 J_2 \cdots J_L$ times the gradient at the output.

![Backward pass through L layers](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/depth_chain.png)

*Backward pass: the gradient is multiplied by one factor per layer.*

- **Factors below 1:** the product shrinks toward 0 (**vanishing gradient**), for example $0.9^{50} \approx 0.005$. The first layers get almost no gradient and learn very slowly.
- **Factors above 1:** the product grows (**exploding gradient**), for example $1.1^{50} \approx 117$. Updates become huge and training diverges.

The activation sets part of the factor:

- The derivative of the sigmoid is at most 0.25, so each sigmoid layer shrinks the gradient by at least a factor 4 before the weights are counted.
- The derivative of tanh is at most 1 (it equals 1 only at 0).
- The derivative of ReLU is 0 or 1.

### Initialization scale

The weights set the other part of the factor. If they start too small, the factors are below 1; if too large, above 1.

The initial variance is chosen so that the signal keeps about the same size from layer to layer. Let $n_\text{in}$ and $n_\text{out}$ be the number of inputs and outputs of the layer (its fan-in and fan-out).

- **Glorot / Xavier:** variance $2/(n_\text{in} + n_\text{out})$. For tanh and sigmoid.
- **He:** variance $2/n_\text{in}$. For ReLU, which sets half of the units to 0 and so needs twice the variance.

Framework defaults:

- A Keras `Dense` layer uses `glorot_uniform`.
- A PyTorch `nn.Linear` uses `kaiming_uniform_` with `a=sqrt(5)`.
- That gives weights uniform in $\pm 1/\sqrt{n_\text{in}}$, with variance $1/(3\,n_\text{in})$: smaller than He.

![Gradient size per layer](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/depth_grad_layers.png)

*Gradient size per layer in a 24-layer tanh network of width 16: plain with small init, plain with Glorot init, residual with small init.*

Experiment: 24 tanh layers of width 16. The bars show the gradient size at each layer relative to the last layer.

- Plain network with small initial weights: the first layer gets $4.8 \times 10^{-7}$ of the last layer's gradient.
- Plain network with Glorot initialization: about 0.7.
- Residual network with the same small initialization: about 4.6.

### Two fixes you will see later in the course

Here is only the idea of each, without derivations.

**Residual connections.** A residual block outputs $x + F(x)$, where $F$ is a few layers. The derivative of the output with respect to $x$ is $1 + dF/dx$, so the gradient has a direct path back through the "+" and does not need to pass through $F$.

![A residual block](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/depth_residual.png)

*A residual block: output = x + F(x), with a skip connection that copies x.*

**BatchNorm.** For each unit, BatchNorm subtracts the mean over the batch and divides by the standard deviation over the batch, so the normalized values have mean 0 and standard deviation 1. Then it scales and shifts them with learned parameters $\gamma$ and $\beta$.

After that, the mean and spread of each unit's input are set by these two learned parameters instead of by the earlier weights, which keeps the per-layer factor under control.

![BatchNorm on one unit](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/depth_batchnorm.png)

*BatchNorm: one unit over a batch is normalized to mean 0 and std 1, then scaled and shifted.*

Interactive demos: [signal tower](https://dbertazioli.github.io/aml-labs/lab1/deep/#tower) (signal through a deep stack of layers) and [plain vs residual](https://dbertazioli.github.io/aml-labs/lab1/deep/#playground).

### References

- Vanishing and exploding gradients: [Deep Learning book, section 8.2.4](https://www.deeplearningbook.org/contents/optimization.html) (cliffs and exploding gradients), [section 6.3.2](https://www.deeplearningbook.org/contents/mlp.html) (logistic sigmoid and hyperbolic tangent), [Pascanu, Mikolov, Bengio, On the difficulty of training recurrent neural networks (2013)](https://proceedings.mlr.press/v28/pascanu13.html)
- Glorot / Xavier initialization: [Glorot & Bengio, Understanding the difficulty of training deep feedforward neural networks (2010)](https://proceedings.mlr.press/v9/glorot10a.html)
- He initialization: [He et al., Delving deep into rectifiers (2015)](https://arxiv.org/abs/1502.01852)
- Initialization in general: [Deep Learning book, section 8.4](https://www.deeplearningbook.org/contents/optimization.html)
- Framework defaults: [Keras docs, Dense](https://keras.io/api/layers/core_layers/dense/), [PyTorch docs, nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
- Residual connections: [He et al., Deep residual learning for image recognition (2016)](https://arxiv.org/abs/1512.03385)
- BatchNorm: [Ioffe & Szegedy, Batch normalization (2015)](https://arxiv.org/abs/1502.03167), [Deep Learning book, section 8.7.1](https://www.deeplearningbook.org/contents/optimization.html)

## Optional: tensors and gradients in Keras

The PyTorch notebook 1 covers the same ideas in its main sections; `fit` does all of this for you.

### Tensors

A tensor is an n-dimensional array, like a numpy array, that can also run on a GPU and be differentiated. Its key properties are `shape` and `dtype`.

`keras.ops` has numpy-like functions for tensors (`ops.zeros`, `ops.arange`, `ops.mean`, ...). TensorFlow tensors are **immutable**: operations return new tensors.

In [ ]:
import tensorflow as tf
from keras import ops

a = ops.zeros((2, 3))                 # shape (2, 3), dtype float32
b = ops.reshape(ops.arange(6), (2, 3))   # integer dtype
c = keras.random.normal((2, 3))        # standard normal
print(a.shape, a.dtype)
print(b.dtype, "->", ops.cast(b, "float32").dtype)   # convert dtype

### Broadcasting

When shapes differ, size-1 (or missing) dimensions are stretched to match.

- `(3, 4) - (4,)` subtracts the vector from every row.
- `(3, 4) - (3, 1)` subtracts one number from each row.
- `keepdims=True` keeps a reduced dimension as size 1.

*Further reading: [NumPy docs, Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) (`keras.ops` follows the same rules)*

In [ ]:
x = ops.reshape(ops.arange(12, dtype="float32"), (3, 4))
print(x - ops.mean(x, axis=0))                 # (3,4) - (4,)
print(x - ops.mean(x, axis=1, keepdims=True))  # (3,4) - (3,1)

> **Exercise A.1** Create a float32 tensor `t` of shape `(3, 4)` containing `0, 1, ..., 11`, row by row.

In [ ]:
t = None  # TODO

In [ ]:
assert t.shape == (3, 4) and t.dtype == "float32" and float(t[2, 3]) == 11

### Standardizing columns

Standardizing a column means computing $z = (x - \mu) / \sigma$, with the column mean $\mu$ and standard deviation $\sigma$.

`ops.std(t, axis=0)` gives $\sigma$ for every column. It is the population std (it divides by $n$), unlike `torch.std`.

*Further reading: [Keras docs, ops.std](https://keras.io/api/ops/numpy/#std-function), [PyTorch docs, torch.std](https://pytorch.org/docs/stable/generated/torch.std.html)*

> **Exercise A.2** Standardize the columns of `t` using broadcasting and no loops. Store the result in `z`.

In [ ]:
z = None  # TODO

In [ ]:
assert z.shape == (3, 4)
assert np.allclose(ops.mean(z, axis=0), 0, atol=1e-6)
assert np.allclose(ops.std(z, axis=0), 1, atol=1e-5)

### numpy interop

- `ops.convert_to_tensor(arr)` turns a numpy array into a tensor.
- `ops.convert_to_numpy(t)` turns a tensor into a numpy array.
- Treat the result as a copy; do not rely on shared memory.

> **Exercise A.3** Make `arr` the numpy array `[1., 2., 3.]` and `s` the tensor built from it.
> Compute `s2 = s * 2` and convert it back to numpy as `back`. Then check that `arr` is unchanged.

In [ ]:
arr = None   # TODO: numpy array [1., 2., 3.]
s = None     # TODO: tensor from arr
s2 = None    # TODO: s * 2
back = None  # TODO: numpy array from s2

In [ ]:
assert np.allclose(arr, [1., 2., 3.]) and np.allclose(back, [2., 4., 6.])

### Gradients with tf.GradientTape

Keras has no gradient API of its own. With the TensorFlow backend you use `tf.GradientTape`.

- Operations on a `tf.Variable` inside `with tf.GradientTape() as tape:` are recorded.
- `tape.gradient(y, x)` returns $dy/dx$.
- Gradients do not accumulate: every `tape.gradient` call starts from scratch.

*Further reading: [TensorFlow guide, Introduction to gradients and automatic differentiation](https://www.tensorflow.org/guide/autodiff)*

In [ ]:
x = tf.Variable(2.0)
with tf.GradientTape() as tape:
    y = x**2 + 3 * x          # dy/dx = 2x + 3 = 7 at x = 2
print(tape.gradient(y, x))

For $f(x) = x^3 + 2x$ the derivative is $f'(x) = 3x^2 + 2$.

> **Exercise A.4** Compute the gradient of $f(x) = x^3 + 2x$ at $x = 2$ with autograd (`g_auto`) and by hand from the derivative (`g_true`).

In [ ]:
x = tf.Variable(2.0)
with tf.GradientTape() as tape:
    f = None  # TODO: f(x)
g_auto = None  # TODO: tape.gradient(...)
g_true = None  # TODO: analytic derivative at x=2 (a float)

In [ ]:
assert abs(float(g_auto) - g_true) < 1e-6 and abs(g_true - 14) < 1e-6

### The tape only watches variables

If the input is a plain tensor or constant, `tape.gradient` returns `None`. There is no error, so this is easy to miss.

- Use a `tf.Variable`, or call `tape.watch(tensor)` inside the tape block.

*Further reading: [TensorFlow guide, controlling what the tape watches](https://www.tensorflow.org/guide/autodiff#controlling_what_the_tape_watches)*

In [ ]:
w = tf.constant(1.0)
with tf.GradientTape() as tape:
    loss_w = 3 * w
print("gradient:", tape.gradient(loss_w, w))   # None: w is not watched

> **Exercise A.5** Redo the computation above so that the gradient is 3: make the tape watch `w` with `tape.watch`.

In [ ]:
w = tf.constant(1.0)
with tf.GradientTape() as tape:
    # TODO: tell the tape to watch w
    loss_w = 3 * w
seen = tape.gradient(loss_w, w)

In [ ]:
assert seen is not None and float(seen) == 3.0